# PV Imports with More

Written by ChatGPT with help from Scott Feister on June 18 2026

The goal is to get our PVAccess trace... with all its metadata like uniqueId and shot nubmer

# Imports

In [1]:
from pprint import pprint

from bluesky import plans as bp
from bluesky.run_engine import RunEngine, autoawait_in_bluesky_event_loop
from ophyd_async.core import init_devices

from pva_ntndarray_source_v2 import PvaNtNdArraySource

# RunEnginer

In [2]:
RE = RunEngine(call_returns_result=True)

# Makes ophyd-async awaitable calls work naturally in Jupyter.
autoawait_in_bluesky_event_loop()

# create device

In [3]:
PV = "ELECTRON-DAQ:trace"

with init_devices():
    electron_daq = PvaNtNdArraySource(
        pv=PV,
        name="electron_daq",
    )

# inspect what Bluesky thinks this device produces

In [4]:
desc = await electron_daq.describe()
pprint(desc)

OrderedDict([('electron_daq_trace',
              {'dtype': 'array',
               'shape': [100],
               'source': 'pva://ELECTRON-DAQ:trace'}),
             ('electron_daq_timeStamp',
              {'dtype': 'number',
               'shape': [],
               'source': 'pva://ELECTRON-DAQ:trace'}),
             ('electron_daq_uniqueId',
              {'dtype': 'integer',
               'shape': [],
               'source': 'pva://ELECTRON-DAQ:trace'}),
             ('electron_daq_dataTimeStamp',
              {'dtype': 'number',
               'shape': [],
               'source': 'pva://ELECTRON-DAQ:trace'}),
             ('electron_daq_shot_num',
              {'dtype': 'integer',
               'shape': [],
               'source': 'pva://ELECTRON-DAQ:trace'}),
             ('electron_daq_start_shot_num',
              {'dtype': 'integer',
               'shape': [],
               'source': 'pva://ELECTRON-DAQ:trace'}),
             ('electron_daq_trace_nt',
           

# read one atomic payload

In [5]:
reading = await electron_daq.read()
pprint(reading)

OrderedDict([('electron_daq_trace',
              {'timestamp': 1781821884.6399994,
               'value': array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   4,   0,   0,   0,   0,   0,   0,   2,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   1,   2,   6,  10,  17,
        22,  26,  33,  39,  41,  44,  49,  49,  50,  54,  58,  58,  61,
        64,  61,  60,  68,  74,  75,  77,  84,  83,  84,  89,  95,  91,
        92, 100, 100,  99, 106, 111, 106, 108, 116, 115, 114, 121, 128,
       119, 125, 134, 136, 130, 137, 143, 139, 141], dtype=uint16)}),
             ('electron_daq_timeStamp',
              {'timestamp': 1781821884.6399994, 'value': 1781821884.6399994}),
             ('electron_daq_uniqueId',
              {'timestamp': 1781821884.6399994, 'value': 6991}),
             ('electron_daq_dataTimeStamp',
              {'timestamp': 1781821884.6399994, 'va

## quick trace check

In [8]:
trace = reading["electron_daq_trace"]["value"]

print("trace type: ", type(trace))
print("trace shape:", trace.shape)
print("trace dtype:", trace.dtype)
print("first 10:   ", trace[-10:])

trace type:  <class 'numpy.ndarray'>
trace shape: (100,)
trace dtype: uint16
first 10:    [128 119 125 134 136 130 137 143 139 141]


# quick metadata check

In [9]:
for key, item in reading.items():
    if key != "electron_daq_trace":
        print(f"{key}: {item['value']}")

electron_daq_timeStamp: 1781821884.6399994
electron_daq_uniqueId: 6991
electron_daq_dataTimeStamp: 1781821884.6399994
electron_daq_shot_num: 6991
electron_daq_start_shot_num: 706638
electron_daq_trace_nt: 100
electron_daq_dt: 5e-06
electron_daq_trace_dt: 5e-06
electron_daq_trace_ymin: 0.0
electron_daq_trace_ymax: 3.3


## run through Bluesky

In [11]:
RE(bp.count([electron_daq], num=3))

RunEngineResult(run_start_uids=('92eea7e9-c0bb-4e66-98f7-df27ced0ba5b',), plan_result='92eea7e9-c0bb-4e66-98f7-df27ced0ba5b', exit_status='success', interrupted=False, reason='', exception=None)